# Introduction

In this notebook, I evaluate a base (non–fine-tuned) code-fixing model on my dataset to establish a baseline. For each sample, the model receives the task description and the buggy Python code and returns a proposed fix. I then run the generated code in an isolated subprocess and record whether it executes successfully within a time limit. If the first attempt fails (except timeouts), I make a second attempt by providing the runtime error back to the model.

At the end, I compute and print summary metrics (e.g., runtime success rate, attempt usage, and model latency). The in-memory results table can be inspected directly in the notebook and will later be compared with the fine-tuned model using the same evaluation split.

## 1. Setup

In this section, I import the required libraries and configure the runtime.  
This includes standard utilities, dataset handling, loading environment variables (for the OpenRouter API key), a notebook-friendly progress bar, and an SSL trust-store fix on Windows to ensure OpenRouter requests work reliably. I also import the LangChain components used to call the model with system/user messages.


In [14]:
# -----------------------
# Standard library
# -----------------------
import os
import json
import re
import time
import random
import subprocess
import sys
import tempfile
from pathlib import Path

# -----------------------
# Data / datasets
# -----------------------
import pandas as pd
from datasets import Dataset

# -----------------------
# Environment variables
# -----------------------
from dotenv import load_dotenv

# -----------------------
# Progress bar (notebook-friendly)
# -----------------------
from tqdm.auto import tqdm


In [ ]:
# -----------------------
# SSL / certificates 
# -----------------------
import truststore
truststore.inject_into_ssl()

In [ ]:
# -----------------------
# Networking (optional: for catching request-related errors)
# -----------------------
from requests.exceptions import RequestException

# -----------------------
# LLM client (OpenRouter via LangChain)
# -----------------------
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

## 2. Load and split the dataset

Here I load the JSON dataset from the local project directory, set a fixed random seed for reproducibility, and convert the list of samples into a Hugging Face `Dataset`. I then create a train/eval split (15% for evaluation) that will be reused later to compare baseline and fine-tuned performance on the same eval set.


In [4]:
SEED = 42
random.seed(SEED)

PROJECT_ROOT = Path.cwd()  
DATA_PATH = (PROJECT_ROOT.parent / "Datasets" / "final_dataset.json").resolve()

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)


print("Samples:", len(data))
print("Keys:", list(data[0].keys()))

dataset = Dataset.from_list(data)
split = dataset.train_test_split(test_size=0.15, seed=SEED)
train_dataset = split["train"]
eval_dataset  = split["test"]

print("Train:", len(train_dataset))
print("Eval :", len(eval_dataset))

Samples: 582
Keys: ['title', 'description', 'difficulty', 'correct_code', 'incorrect_code', 'error_type']
Train: 494
Eval : 88


## 3. Configure the model client

In this cell, I load the OpenRouter API key from a local `.env` file (so it is not stored in the notebook), configure the OpenRouter endpoint and model ID, and initialize a LangChain `ChatOpenAI` client. I also define `llm_fix()`, a small wrapper that measures request latency and retries with exponential backoff to handle transient connection or rate-limit issues.


In [5]:
load_dotenv()  # loads variables from .env into environment

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found. Check your .env file.")

OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
MODEL_ID = os.getenv("OPENROUTER_MODEL", "qwen/qwen2.5-coder-7b-instruct")

APP_REFERER = "http://localhost"
APP_TITLE = "code-fixer-eval"

llm = ChatOpenAI(
    model=MODEL_ID,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    temperature=0.0,
    max_tokens=7000,
    default_headers={
        "HTTP-Referer": APP_REFERER,
        "X-Title": APP_TITLE,
    },
)

def llm_fix(messages, retries=6, min_backoff=1.0, max_backoff=20.0):
    """
    Calls llm.invoke(messages) with retry/backoff for connection/rate/transient errors.
    Returns (content, latency_seconds).
    """
    t0 = time.perf_counter()
    last_err = None

    for attempt in range(retries):
        try:
            resp = llm.invoke(messages)
            t1 = time.perf_counter()
            return resp.content, (t1 - t0)

        except Exception as e:
            # Catch common transient errors broadly (APIConnectionError, Timeout, 429, 5xx)
            last_err = e

            # backoff + jitter
            sleep_s = min(max_backoff, min_backoff * (2 ** attempt)) + random.random()
            print(f"[WARN] LLM call failed ({type(e).__name__}): {e} | retrying in {sleep_s:.1f}s...")
            time.sleep(sleep_s)

    raise last_err

## 4. Prompt templates

This section defines the system instruction and the user prompt format used to ask the model to fix buggy AI/ML Python code. I run two attempts per sample: the first uses only the task + buggy code, and the second (if needed) includes the runtime error message to guide a minimal correction.


In [6]:
# -----------------------
# Prompts
# -----------------------
SYSTEM_TEXT = (
    "You are a Python code fixer for AI/ML projects. "
    "Fix the provided code so it runs end-to-end and matches the task requirements. "
    "Rules: make the smallest possible change(s); do NOT add new features, demo blocks, or extra prints; "
    "do NOT refactor or rename unless required. "
    "Output ONLY the corrected Python code (no markdown, no explanations)."
)

def build_user_prompt_no_hint(ex):
    return (
        f"Task title: {ex['title']}\n"
        f"Task description: {ex['description']}\n\n"
        "Buggy code:\n"
        "```python\n"
        f"{ex['incorrect_code']}\n"
        "```\n\n"
        "Fix the code. Output ONLY the corrected Python code."
    )

def build_user_prompt_with_error(ex, prev_code, error_text):
    return (
        f"Task title: {ex['title']}\n"
        f"Task description: {ex['description']}\n\n"
        "The following code still fails at runtime with this error:\n"
        f"{error_text}\n\n"
        "Code:\n"
        "```python\n"
        f"{prev_code}\n"
        "```\n\n"
        "Fix ONLY what is necessary to remove the error and satisfy the task. "
        "Do not add new sections. Output ONLY the corrected Python code."
    )


## 5. Helper utilities

This section contains utility functions used during evaluation. It includes a robust extractor to ensure the model output is treated as pure Python code, a sandboxed subprocess runner with timeouts and safe decoding, parsers for the dataset’s expected error metadata, and small helpers for classifying runtime failures and selecting per-sample time limits.


In [ ]:
# -----------------------
# Helpers
# -----------------------

_CODEBLOCK_RE = re.compile(r"```(?:python)?\s*(.*?)```", re.DOTALL | re.IGNORECASE)

def extract_python_code(text: str) -> str:
    """
    Best-effort extraction of pure Python code from an LLM response.
    Priority:
      1) content inside ```python ... ```
      2) otherwise, remove obvious non-code leading lines until we hit code-like content
    """
    text = (text or "").strip()

    # 1) Prefer fenced code if present
    m = _CODEBLOCK_RE.search(text)
    if m:
        return m.group(1).strip()

    # 2) Remove leading junk lines until it looks like code
    lines = text.splitlines()

    def looks_like_code(line: str) -> bool:
        s = line.strip()
        if not s:
            return False
        # common python starters
        if s.startswith(("import ", "from ", "def ", "class ", "@", "if ", "for ", "while ", "try:", "with ", "print(")):
            return True
        # assignments / function calls
        if re.match(r"^[A-Za-z_][A-Za-z0-9_]*\s*=", s):
            return True
        if re.match(r"^[A-Za-z_][A-Za-z0-9_]*\(", s):
            return True
        # comments can be part of code
        if s.startswith("#"):
            return True
        return False

    start = 0
    for i, line in enumerate(lines):
        if looks_like_code(line):
            start = i
            break
    cleaned = "\n".join(lines[start:]).strip()

    # 3) If model echoed "python" label, remove it
    cleaned = re.sub(r"^\s*python\s*\n", "", cleaned, flags=re.IGNORECASE)

    return cleaned


def run_python(code: str, timeout=25, debug=False):
    code = extract_python_code(code)

    with tempfile.TemporaryDirectory() as td:
        tmp = Path(td) / "_tmp_run_eval.py"
        tmp.write_text(code, encoding="utf-8")

        cmd = [sys.executable, str(tmp)]
        if debug:
            print("CMD:", cmd)
            print("TMP exists:", tmp.exists(), "size:", tmp.stat().st_size)

        try:
            proc = subprocess.run(
                cmd,
                stdout=subprocess.PIPE,        # capture stdout for debugging
                stderr=subprocess.PIPE,
                text=True,
                encoding="utf-8",
                errors="replace",
                timeout=timeout,
            )
            ok = (proc.returncode == 0)
            err = (proc.stderr or "")
            err_tail = "\n".join(err.splitlines()[-25:])
            if debug and not ok:
                print("STDOUT tail:\n", "\n".join((proc.stdout or "").splitlines()[-25:]))
                print("STDERR tail:\n", err_tail)
            return ok, err_tail

        except subprocess.TimeoutExpired:
            return False, f"TimeoutExpired: exceeded {timeout}s"
        except Exception as e:
            # show the real reason
            return False, f"RunnerError: {type(e).__name__}: {e}"
        
        
def parse_error_type_field(error_type_text: str):
    """
    Field example:
      "NameError: ... || line: y_pred = best_modell.predict(X_test)"
    Returns:
      expected_exception="NameError" (or None)
      expected_line="y_pred = best_modell.predict(X_test)" (or None)
    """
    if not error_type_text:
        return None, None

    expected_exception = None
    expected_line = None

    m = re.match(r"\s*([A-Za-z_][A-Za-z0-9_]*)\s*:", error_type_text)
    if m:
        expected_exception = m.group(1)

    m2 = re.search(r"\|\|\s*line:\s*(.*)\s*$", error_type_text)
    if m2:
        expected_line = m2.group(1).strip()

    return expected_exception, expected_line


def contains_expected_bug_line(code: str, expected_line: str):
    if not expected_line:
        return None
    code = extract_python_code(code)
    return expected_line.strip() in code


def timeout_for(difficulty: str) -> int:
    d = (difficulty or "").lower()
    if d == "easy":
        return 120
    if d == "medium":
        return 180
    return 300

import re

import re

def extract_runtime_exception_name(stderr_tail: str):
    if not stderr_tail:
        return None
    if "TimeoutExpired" in stderr_tail:
        return "TimeoutExpired"
    m = re.findall(
        r"([A-Za-z_][A-Za-z0-9_]*(?:Error|Exception))\s*:\s*",
        stderr_tail
    )
    return m[-1] if m else None


## 6. Select evaluation samples

Here I randomly sample a fixed number of items from the evaluation split (capped by the eval set size). Since I use a fixed seed and reuse the same `sample_indices`, the **exact same eval subset** is used across models (e.g., Qwen and Mistral), ensuring a fair and consistent comparison.


In [ ]:
# -----------------------
# Choose eval samples
# -----------------------
N_SAMPLES = 88
N_SAMPLES = min(N_SAMPLES, len(eval_dataset))
sample_indices = random.sample(range(len(eval_dataset)), N_SAMPLES)

## 7. Baseline evaluation loop

In this cell, I iterate over the selected eval samples and ask the model to repair each buggy program. I run the model output in an isolated subprocess and record runtime success, error type (or timeout), and LLM latency. If the first attempt fails for a non-timeout reason, I run a second attempt by feeding the runtime error back to the model. Each result is stored in `rows` for later summarization and comparison with other models.


In [ ]:
# -----------------------
# Main evaluation loop (VS Code notebook friendly)
# -----------------------
rows = []

for pos, idx in enumerate(tqdm(sample_indices, desc="Evaluating")):
    ex = eval_dataset[int(idx)]

    exp_exc, exp_line = parse_error_type_field(ex.get("error_type", ""))
    total_llm_time = 0.0

    # ---- Attempt 1
    msgs1 = [
        SystemMessage(content=SYSTEM_TEXT),
        HumanMessage(content=build_user_prompt_no_hint(ex)),
    ]
    pred1, t1 = llm_fix(msgs1)
    total_llm_time += t1
    pred1_clean = extract_python_code(pred1)

    ok1, err1 = run_python(pred1_clean, timeout=timeout_for(ex.get("difficulty", "")))
    err1_type = extract_runtime_exception_name(err1)
    still1 = contains_expected_bug_line(pred1_clean, exp_line)

    # Success on attempt 1
    if ok1:
        rows.append({
            "idx": int(idx),
            "title": ex.get("title", ""),
            "difficulty": ex.get("difficulty", ""),
            "expected_exception": exp_exc,
            "expected_line": exp_line,
            "attempt_used": 1,
            "llm_time_sec": total_llm_time,
            "runtime_ok": True,
            "final_error_tail": "",
            "err1_type": err1_type,
            "err2_type": None,
            "still_bug_line_after_1": still1,
            "still_bug_line_after_2": None,
            "fixed_expected_bug_after_2": True,
            "final_prediction": pred1_clean,
        })
        tqdm.write(f"[OK] {pos+1}/{len(sample_indices)} idx={idx} attempt=1 llm={total_llm_time:.2f}s expected={exp_exc}")
        continue

    # ---- NEW: skip Attempt 2 if Attempt 1 timed out
    if err1_type == "TimeoutExpired":
        rows.append({
            "idx": int(idx),
            "title": ex.get("title", ""),
            "difficulty": ex.get("difficulty", ""),
            "expected_exception": exp_exc,
            "expected_line": exp_line,
            "attempt_used": 1,  # we only tried once
            "llm_time_sec": total_llm_time,
            "runtime_ok": False,
            "final_error_tail": err1,  # store timeout message
            "err1_type": err1_type,
            "err2_type": None,
            "still_bug_line_after_1": still1,
            "still_bug_line_after_2": None,
            "fixed_expected_bug_after_2": None,
            "final_prediction": pred1_clean,
        })
        tqdm.write(
            f"[TIMEOUT] {pos+1}/{len(sample_indices)} idx={idx} attempt=1 llm={total_llm_time:.2f}s "
            f"expected={exp_exc} still_bug_line={still1}"
        )
        continue

    # ---- Attempt 2 (only if attempt 1 failed for a non-timeout reason)
    msgs2 = [
        SystemMessage(content=SYSTEM_TEXT),
        HumanMessage(content=build_user_prompt_with_error(ex, pred1_clean, err1)),
    ]
    pred2, t2 = llm_fix(msgs2)
    total_llm_time += t2
    pred2_clean = extract_python_code(pred2)

    ok2, err2 = run_python(pred2_clean, timeout=timeout_for(ex.get("difficulty", "")))
    err2_type = extract_runtime_exception_name(err2)
    still2 = contains_expected_bug_line(pred2_clean, exp_line)

    # heuristic: did we fix the expected bug?
    if ok2:
        fixed_flag = True
    else:
        fixed_flag = (still2 is False) and (err2_type != exp_exc)

    rows.append({
        "idx": int(idx),
        "title": ex.get("title", ""),
        "difficulty": ex.get("difficulty", ""),
        "expected_exception": exp_exc,
        "expected_line": exp_line,
        "attempt_used": 2,
        "llm_time_sec": total_llm_time,
        "runtime_ok": bool(ok2),
        "final_error_tail": "" if ok2 else err2,
        "err1_type": err1_type,
        "err2_type": err2_type,
        "still_bug_line_after_1": still1,
        "still_bug_line_after_2": still2,
        "fixed_expected_bug_after_2": fixed_flag,
        "final_prediction": pred2_clean,
    })

    status = "OK" if ok2 else "FAIL"
    tqdm.write(
        f"[{status}] {pos+1}/{len(sample_indices)} idx={idx} attempt=2 llm={total_llm_time:.2f}s "
        f"expected={exp_exc} err1={err1_type} err2={err2_type} still_bug_line={still2}"
    )


Evaluating:   1%|          | 1/88 [02:04<3:00:16, 124.33s/it]

[TIMEOUT] 1/88 idx=81 attempt=1 llm=4.13s expected=AttributeError still_bug_line=True


Evaluating:   2%|▏         | 2/88 [05:13<3:52:58, 162.54s/it]

[TIMEOUT] 2/88 idx=14 attempt=1 llm=9.19s expected=NameError still_bug_line=False


Evaluating:   3%|▎         | 3/88 [05:26<2:13:40, 94.36s/it] 

...

[OK] 86/88 idx=33 attempt=1 llm=3.74s expected=AttributeError


Evaluating:  99%|█████████▉| 87/88 [54:34<00:16, 16.06s/it]

[OK] 87/88 idx=63 attempt=1 llm=6.20s expected=NameError


Evaluating: 100%|██████████| 88/88 [54:46<00:00, 37.35s/it]

[OK] 88/88 idx=59 attempt=1 llm=3.35s expected=KeyError


In [15]:
# -----------------------
# Summary + Save (VS Code)
# -----------------------
df = pd.DataFrame(rows)

print("\n=== Summary ===")
print("Total samples:", len(df))
print("Runtime success rate:", round(100 * df["runtime_ok"].mean(), 2), "%")
print("Used attempt=1:", int((df["attempt_used"] == 1).sum()))
print("Used attempt=2:", int((df["attempt_used"] == 2).sum()))
print("Avg LLM time:", round(df["llm_time_sec"].mean(), 2), "sec")


=== Summary ===
Total samples: 88
Runtime success rate: 53.41 %
Used attempt=1: 48
Used attempt=2: 40
Avg LLM time: 10.08 sec


## Conclusion

On the selected evaluation subset (88 samples), the baseline model achieved a **53.41%** runtime success rate. Out of all samples, **48** were fixed on the first attempt, while **40** required a second attempt using runtime error feedback, indicating that error-guided prompting provides a noticeable boost but does not fully resolve all failures. The average model response latency was **10.08 seconds** per sample (including both attempts when used). These results establish the baseline performance that I will compare against the fine-tuned model using the same evaluation split and sample subset.
